In [ ]:
"""
=============================================================================
Baseline Random Forest Classifier for Cancer Cachexia Biomarker Identification
=============================================================================

Dataset  : GSE75473 – Human skeletal muscle miRNA + mRNA (cancer cachexia)
Reference: Narasimhan et al., 2017, PMID 28058815

CHANGE vs original: Biomarker ranking, plots, and summary now report
miRNA-ONLY features (filtered by omic == "miRNA") instead of mixing
all feature types. A full-feature ranking CSV is also saved separately.

Pipeline stages:
1.  Data loading & label engineering
2.  miRNA pre-processing
3.  mRNA pre-processing
4.  Differential expression analysis (Welch t-test + BH FDR)
5.  miRNA–mRNA graph construction + graph feature extraction
6.  Sample feature matrix assembly
7.  Class-imbalance handling (SMOTE)
8.  Random Forest Classifier (with GridSearchCV inner tuning)
9.  Nested cross-validation (outer 5-fold / inner 3-fold)
10. Evaluation: AUROC, AUPRC, F1, MCC, sensitivity, specificity
11. miRNA Biomarker ranking (feature_importances_ — Gini impurity)
12. Result export + comparison table vs attention model

Run
───
    python cachexia_rf_baseline.py
"""

# ── stdlib ─────────────────────────────────────────────────────────────────
import os, warnings, time
warnings.filterwarnings("ignore")

# ── third-party ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, matthews_corrcoef,
                              confusion_matrix, roc_curve,
                              precision_recall_curve)
from imblearn.over_sampling import SMOTE
from matplotlib.patches import Patch

# ═══════════════════════════════════════════════════════════════════════════
# 0.  CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════
DATA_DIR    = "/content/data_dir"
OUT_DIR     = "/content/data_dir/outputs_rf_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

MIRNA_FILE  = os.path.join(DATA_DIR, "GSE75473_miR_RPKM_normalised_counts_CC.txt")
MRNA_FILE   = os.path.join(DATA_DIR, "GSE75473_norm_counts_TPM_GRCh38.p13_NCBI (1).tsv")
ANNOT_FILE  = os.path.join(DATA_DIR, "Human.GRCh38.p13.annot.tsv")
SERIES_FILE = os.path.join(DATA_DIR, "GSE75473_series_matrix.txt")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DE_PVAL_THR = 0.05
FC_THR      = 1.4
CORR_THR    = -0.3
TOP_MRNA    = 100
TOP_MIRNA   = 30

OUTER_CV    = 5
INNER_CV    = 3

# ── RF hyper-param search space ────────────────────────────────────────────
RF_PARAM_GRID = {
    "n_estimators":      [100, 300, 500],
    "max_depth":         [None, 5, 10],
    "min_samples_split": [2, 5],
    "max_features":      ["sqrt", "log2"],
    "class_weight":      ["balanced"],
}

# ── Optional: skip DE+graph by loading pre-saved numpy arrays ──────────────
LOAD_PRECOMPUTED   = False
PRECOMPUTED_X_PATH = os.path.join(DATA_DIR, "X_features.npy")
PRECOMPUTED_Y_PATH = os.path.join(DATA_DIR, "y_labels.npy")
PRECOMPUTED_NAMES  = os.path.join(DATA_DIR, "feature_names.txt")

print("=" * 70)
print("  Random Forest Baseline – Cancer Cachexia (GSE75473)")
print("  [miRNA-focused biomarker output]")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════
def parse_series_matrix(path):
    titles, gsms = [], []
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!Sample_title"):
                parts = line.strip().split("\t")[1:]
                titles = [p.strip('"') for p in parts]
            elif line.startswith("!Sample_geo_accession"):
                parts = line.strip().split("\t")[1:]
                gsms = [p.strip('"') for p in parts]
    labels = {}
    for gsm, title in zip(gsms, titles):
        t = title.lower()
        if "cachectic cases" in t:
            labels[gsm] = 1
        elif "non-cachectic" in t:
            labels[gsm] = 0
        elif "cachectic controls" in t:
            labels[gsm] = 1
    return labels


def bh_correction(pvals):
    """Benjamini-Hochberg FDR, clamped to [0, 1]."""
    n = len(pvals)
    ranks = np.argsort(pvals)
    bh = np.empty(n)
    for i, r in enumerate(ranks):
        bh[r] = pvals[r] * n / (i + 1)
    bh = np.minimum.accumulate(bh[::-1])[::-1]
    return np.minimum(bh, 1.0)


def de_analysis(expr_df, labels, fc_thr=FC_THR, pval_thr=DE_PVAL_THR, omic="feature"):
    cache_idx = labels[labels == 1].index
    ctrl_idx  = labels[labels == 0].index
    cache_df  = expr_df.loc[cache_idx]
    ctrl_df   = expr_df.loc[ctrl_idx]

    results = []
    for feat in expr_df.columns:
        g1 = cache_df[feat].dropna().values
        g0 = ctrl_df[feat].dropna().values
        if len(g1) < 2 or len(g0) < 2:
            continue
        t, p = ss.ttest_ind(g1, g0, equal_var=False)
        fc = g1.mean() - g0.mean()
        results.append({"feature": feat, "log2FC": fc, "pval": p})

    res_df = pd.DataFrame(results).set_index("feature")
    res_df["padj"] = bh_correction(res_df["pval"].values)
    res_df["sig"]  = (res_df["padj"] < pval_thr) & (res_df["log2FC"].abs() > np.log2(fc_thr))
    sig_n = res_df["sig"].sum()
    print(f"   {omic}: {sig_n} DE features (padj<{pval_thr}, |FC|>{fc_thr})")
    return res_df


def get_omic_type(feat_name, gene_sym):
    """Classify feature as miRNA / mRNA / graph_feature."""
    s = str(feat_name)
    if ("mir" in s.lower()) and ("graph" not in s):
        return "miRNA"
    if "graph" in s:
        return "graph_feature"
    sym = gene_sym.get(s, "")
    return f"mRNA({sym})" if sym else "mRNA"


def is_mirna(feat_name):
    s = str(feat_name).lower()
    return ("mir" in s) and ("graph" not in s)

# ═══════════════════════════════════════════════════════════════════════════
# 1–7.  DATA PREPARATION
# ═══════════════════════════════════════════════════════════════════════════

if LOAD_PRECOMPUTED and os.path.exists(PRECOMPUTED_X_PATH):
    print("\n[1-7] Loading pre-computed feature matrix …")
    X = np.load(PRECOMPUTED_X_PATH)
    y = np.load(PRECOMPUTED_Y_PATH)
    with open(PRECOMPUTED_NAMES) as f:
        feature_names = [l.strip() for l in f.readlines()]
    print(f"   X={X.shape}, y classes={np.bincount(y)}")
    gene_sym = {}
    mir_de = mrna_de = None

else:
    # ── 1. Labels ──────────────────────────────────────────────────────────
    print("\n[1] Parsing sample labels …")
    sample_labels = parse_series_matrix(SERIES_FILE)
    label_series  = pd.Series(sample_labels, name="label")
    print(f"   Samples: {len(label_series)}  |  "
          f"Cachexia: {label_series.sum()}  |  "
          f"Control: {(label_series==0).sum()}")

    # ── 2. miRNA ───────────────────────────────────────────────────────────
    print("\n[2] Loading miRNA RPKM data …")
    mir_raw    = pd.read_csv(MIRNA_FILE, sep="\t")
    sample_col = mir_raw.columns[0]
    attrib_col = mir_raw.columns[2]
    mir_exprs  = mir_raw.drop(columns=[mir_raw.columns[1], attrib_col])
    mir_exprs  = mir_exprs.set_index(sample_col)
    numeric_to_gsm = {}
    with open(SERIES_FILE, "r", encoding="utf-8", errors="replace") as fh:
        titles_line, gsm_line = "", ""
        for line in fh:
            if line.startswith("!Sample_title"):
                titles_line = line.strip()
            elif line.startswith("!Sample_geo_accession"):
                gsm_line = line.strip()
        for t, g in zip(titles_line.split("\t")[1:], gsm_line.split("\t")[1:]):
            t, g = t.strip('"'), g.strip('"')
            if "[" in t:
                num = t.split("[")[1].split("]")[0]
                numeric_to_gsm[num] = g

    mir_exprs.index = [str(i) for i in mir_exprs.index]
    mir_exprs.index = [numeric_to_gsm.get(i, i) for i in mir_exprs.index]
    common_mir      = mir_exprs.index.intersection(label_series.index)
    mir_exprs       = mir_exprs.loc[common_mir].astype(float)
    mir_labels      = label_series.loc[common_mir]
    mir_log         = np.log2(mir_exprs + 1)
    print(f"   After label alignment: {mir_log.shape}")

    # ── 3. mRNA ────────────────────────────────────────────────────────────
    print("\n[3] Loading mRNA TPM data …")
    mrna_raw  = pd.read_csv(MRNA_FILE, sep="\t", index_col=0)
    annot     = pd.read_csv(ANNOT_FILE, sep="\t", dtype=str).set_index("GeneID")
    gene_sym  = annot["Symbol"].to_dict()

    if "GeneType" in annot.columns:
        coding_ids = annot[annot["GeneType"].isin(
            ["protein-coding", "ncRNA", "lncRNA"])].index.astype(str)
        mrna_filt  = mrna_raw.loc[mrna_raw.index.astype(str).isin(coding_ids)]
    else:
        mrna_filt  = mrna_raw.copy()

    mrna_T      = mrna_filt.T
    common_m    = mrna_T.index.intersection(label_series.index)
    mrna_T      = mrna_T.loc[common_m].astype(float)
    mrna_labels = label_series.loc[common_m]
    expr_mask   = mrna_T.mean(axis=0) >= 1
    mrna_T      = mrna_T.loc[:, expr_mask]
    mrna_log    = np.log2(mrna_T + 1)
    print(f"   After filtering: samples={mrna_log.shape[0]}, genes={mrna_log.shape[1]}")

    # ── 4. DE analysis ─────────────────────────────────────────────────────
    print("\n[4] Differential expression analysis …")
    mir_de  = de_analysis(mir_log,  mir_labels,  omic="miRNA")
    mrna_de = de_analysis(mrna_log, mrna_labels, omic="mRNA")

    mir_sig  = mir_de[mir_de["sig"]].sort_values("log2FC", key=abs, ascending=False)
    mrna_sig = mrna_de[mrna_de["sig"]].sort_values("log2FC", key=abs, ascending=False)
    if len(mir_sig) < 5:
        mir_sig  = mir_de.sort_values("pval").head(TOP_MIRNA)
        print(f"   miRNA fallback: top-{TOP_MIRNA} by p-value")
    if len(mrna_sig) < 5:
        mrna_sig = mrna_de.sort_values("pval").head(TOP_MRNA)
        print(f"   mRNA fallback: top-{TOP_MRNA} by p-value")

    sel_mir  = mir_sig.index[:TOP_MIRNA].tolist()
    sel_mrna = mrna_sig.index[:TOP_MRNA].tolist()
    print(f"   Selected: {len(sel_mir)} miRNAs, {len(sel_mrna)} mRNAs")

    # ── 5. Graph construction ──────────────────────────────────────────────
    print("\n[5] Constructing miRNA–mRNA regulatory graph …")
    common_samples = mir_log.index.intersection(mrna_log.index)
    mir_aligned    = mir_log.loc[common_samples, [m for m in sel_mir  if m in mir_log.columns]]
    mrna_aligned   = mrna_log.loc[common_samples, [g for g in sel_mrna if g in mrna_log.columns]]
    labels_aligned = label_series.loc[common_samples]

    G = nx.Graph()
    for m in mir_aligned.columns:
        G.add_node(m, type="miRNA",
                   log2FC=float(mir_de.loc[m, "log2FC"]) if m in mir_de.index else 0.0)
    for g in mrna_aligned.columns:
        G.add_node(g, type="mRNA",
                   log2FC=float(mrna_de.loc[g, "log2FC"]) if g in mrna_de.index else 0.0)
    edge_count = 0
    for mi in mir_aligned.columns:
        mi_vec = mir_aligned[mi].values
        for g in mrna_aligned.columns:
            g_vec = mrna_aligned[g].values
            if len(mi_vec) < 4 or len(g_vec) < 4:
                continue
            r, _ = ss.pearsonr(mi_vec, g_vec)
            if r <= CORR_THR:
                G.add_edge(mi, g, weight=float(abs(r)))
                edge_count += 1
    print(f"   Edges: {edge_count}  |  Nodes: {G.number_of_nodes()}")

    # ── 6. Graph features ──────────────────────────────────────────────────
    print("\n[6] Extracting graph topology features …")
    deg_cent = nx.degree_centrality(G)
    bet_cent = nx.betweenness_centrality(G, normalized=True)
    try:
        from networkx.algorithms.community import greedy_modularity_communities
        communities = list(greedy_modularity_communities(G))
        node_comm   = {n: i for i, c in enumerate(communities) for n in c}
    except Exception:
        node_comm   = {n: 0 for n in G.nodes()}

    def graph_feature_vec(nodes):
        return pd.DataFrame({n: {"deg_cent":  deg_cent.get(n, 0),
                                 "between":   bet_cent.get(n, 0),
                                 "community": node_comm.get(n, -1)}
                             for n in nodes}).T

    mir_gf  = graph_feature_vec(mir_aligned.columns)
    mrna_gf = graph_feature_vec(mrna_aligned.columns)

    # ── 7. Feature matrix ──────────────────────────────────────────────────
    print("\n[7] Assembling sample feature matrix …")
    X_mir  = mir_aligned.values
    X_mrna = mrna_aligned.values
    n_samp = X_mir.shape[0]

    graph_block = np.tile(
        np.concatenate([
            mir_gf.values.mean(axis=0),  mir_gf.values.max(axis=0),
            mrna_gf.values.mean(axis=0), mrna_gf.values.max(axis=0),
        ]),
        (n_samp, 1)
    )
    X = np.concatenate([X_mir, X_mrna, graph_block], axis=1)
    y = labels_aligned.values.astype(int)

    feature_names = (
        list(mir_aligned.columns) +
        list(mrna_aligned.columns) +
        [f"mir_graph_{s}"  for s in ["deg_mean", "bet_mean", "comm_mean",
                                      "deg_max",  "bet_max",  "comm_max"]] +
        [f"mrna_graph_{s}" for s in ["deg_mean", "bet_mean", "comm_mean",
                                      "deg_max",  "bet_max",  "comm_max"]]
    )
    assert len(feature_names) == X.shape[1]
    print(f"   Feature matrix: {X.shape}  |  Classes: {np.bincount(y)}")

# ── Index positions of miRNA features (used throughout for miRNA-only plots) ─
mirna_feat_indices = [i for i, n in enumerate(feature_names) if is_mirna(n)]
mirna_feat_names   = [feature_names[i] for i in mirna_feat_indices]
print(f"\n   miRNA feature count in matrix: {len(mirna_feat_indices)}")

# ═══════════════════════════════════════════════════════════════════════════
# 8–9.  NESTED CV WITH RANDOM FOREST + SMOTE
# ═══════════════════════════════════════════════════════════════════════════
print("\n[8] Nested CV – Random Forest + SMOTE …")

outer_cv = StratifiedKFold(n_splits=OUTER_CV, shuffle=True, random_state=RANDOM_SEED)
inner_cv = StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_SEED)

fold_metrics        = []
all_probs           = np.zeros(len(y))
all_preds           = np.zeros(len(y), dtype=int)
feat_imp_accum      = np.zeros(X.shape[1])   # accumulates ALL feature importances
best_params_log     = []
fold_importances    = []                      # per-fold for stability analysis

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
    t0 = time.time()
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr, yte = y[train_idx], y[test_idx]

    # ── Scale ──────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(Xtr)
    Xte_s  = scaler.transform(Xte)
# ── SMOTE ─────────────────────────────────────────────────────────────
    k_neighbors = min(5, np.bincount(ytr).min() - 1)
    if k_neighbors < 1:
        Xtr_bal, ytr_bal = Xtr_s, ytr
    else:
        sm = SMOTE(k_neighbors=k_neighbors, random_state=RANDOM_SEED)
        try:
            Xtr_bal, ytr_bal = sm.fit_resample(Xtr_s, ytr)
        except Exception:
            Xtr_bal, ytr_bal = Xtr_s, ytr

    # ── Inner GridSearchCV ─────────────────────────────────────────────────
    rf_base = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)
    gs = GridSearchCV(
        rf_base, RF_PARAM_GRID,
        cv=inner_cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
    )
    gs.fit(Xtr_bal, ytr_bal)
    best_clf = gs.best_estimator_
    best_params_log.append(gs.best_params_)

    # ── Predict ────────────────────────────────────────────────────────────
    prob_te = best_clf.predict_proba(Xte_s)[:, 1]
    pred_te = (prob_te >= 0.5).astype(int)
    all_probs[test_idx] = prob_te
    all_preds[test_idx] = pred_te

    # ── Metrics ────────────────────────────────────────────────────────────
    cm = confusion_matrix(yte, pred_te, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    auc  = roc_auc_score(yte, prob_te) if len(np.unique(yte)) > 1 else 0.5
    prc  = average_precision_score(yte, prob_te) if len(np.unique(yte)) > 1 else 0.5
    f1   = f1_score(yte, pred_te, zero_division=0)
    mcc  = matthews_corrcoef(yte, pred_te)

    fold_metrics.append({
        "fold": fold + 1, "AUROC": auc, "AUPRC": prc,
        "F1": f1, "MCC": mcc, "Sens": sens, "Spec": spec,
        "best_n_est":   gs.best_params_["n_estimators"],
        "best_depth":   str(gs.best_params_["max_depth"]),
        "best_max_feat": gs.best_params_["max_features"],
    })

    # ── Accumulate feature importances ────────────────────────────────────
    feat_imp_accum += best_clf.feature_importances_
    fold_importances.append(best_clf.feature_importances_)

    elapsed = time.time() - t0
    print(f"   Fold {fold+1}: AUROC={auc:.3f}  AUPRC={prc:.3f}  "
          f"F1={f1:.3f}  MCC={mcc:.3f}  "
          f"Sens={sens:.3f}  Spec={spec:.3f}  [{elapsed:.1f}s]  "
          f"best={gs.best_params_['n_estimators']}trees, "
          f"depth={gs.best_params_['max_depth']}")

metrics_df = pd.DataFrame(fold_metrics)
print("\n── RF Baseline Summary (mean ± std) ──")
for col in ["AUROC", "AUPRC", "F1", "MCC", "Sens", "Spec"]:
    print(f"   {col:6s}: {metrics_df[col].mean():.3f} ± {metrics_df[col].std():.3f}")

# ═══════════════════════════════════════════════════════════════════════════
# 10.  BIOMARKER RANKING — miRNA ONLY (Gini importance)
# ═══════════════════════════════════════════════════════════════════════════
print("\n[10] Ranking miRNA biomarkers by Gini importance …")

feat_imp_mean = feat_imp_accum / OUTER_CV

# ── Full ranking (all features) – saved to CSV for reference ───────────────
all_importance_df = pd.DataFrame({
    "feature":    feature_names,
    "importance": feat_imp_mean,
}).sort_values("importance", ascending=False).reset_index(drop=True)
all_importance_df["omic"] = all_importance_df["feature"].apply(
    lambda f: get_omic_type(f, gene_sym)
)

# Attach DE stats if available
if mir_de is not None and mrna_de is not None:
    def get_de_stats(feat_name):
        if feat_name in mir_de.index:
            row = mir_de.loc[feat_name]
            return round(row["log2FC"], 3), round(row["padj"], 4)
        if feat_name in mrna_de.index:
            row = mrna_de.loc[feat_name]
            return round(row["log2FC"], 3), round(row["padj"], 4)
        return 0.0, 1.0

    all_importance_df[["log2FC", "padj"]] = all_importance_df["feature"].apply(
        lambda f: pd.Series(get_de_stats(f))
    )

# ── miRNA-ONLY ranking ────────────────────────────────────────────────────
mirna_importance_df = (
    all_importance_df[all_importance_df["omic"] == "miRNA"]
    .reset_index(drop=True)
)

top20 = mirna_importance_df.head(20)
print("\n  Top miRNA biomarkers (by Gini importance):")
print(top20[["feature", "omic", "importance"]].to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# 11.  SAVE RESULTS
# ═══════════════════════════════════════════════════════════════════════════
print("\n[11] Saving results …")

metrics_df.to_csv(os.path.join(OUT_DIR, "rf_cv_metrics.csv"), index=False)

# miRNA-only ranking (primary output)
mirna_importance_df.to_csv(
    os.path.join(OUT_DIR, "rf_mirna_biomarker_ranking.csv"), index=False
)
# Full ranking kept for reference
all_importance_df.to_csv(
    os.path.join(OUT_DIR, "rf_all_features_ranking.csv"), index=False
)
print("   rf_mirna_biomarker_ranking.csv  ← miRNA only")
print("   rf_all_features_ranking.csv     ← all features (reference)")

# ── Figure 1: ROC + PRC ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Random Forest Baseline – Cachexia Classification", fontsize=13, fontweight="bold")

fpr_all, tpr_all, _ = roc_curve(y, all_probs)
pr_prec, pr_rec, _  = precision_recall_curve(y, all_probs)
overall_auc = roc_auc_score(y, all_probs)
overall_prc = average_precision_score(y, all_probs)

ax = axes[0]
ax.plot(fpr_all, tpr_all, lw=2, color="forestgreen",
        label=f"All folds AUROC={overall_auc:.3f}")
ax.plot([0, 1], [0, 1], '--', color='grey')
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(pr_rec, pr_prec, lw=2, color="darkorange",
        label=f"All folds AUPRC={overall_prc:.3f}")
ax.axhline(y.mean(), linestyle='--', color='grey', label="Baseline")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rf_roc_prc_curves.png"), dpi=150, bbox_inches="tight")
plt.close()
print("   rf_roc_prc_curves.png saved.")

# ── Figure 2: Top-20 miRNA biomarkers ─────────────────────────────────────
# All bars are red because all features shown are miRNAs
fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = ["#E63946"] * len(top20)   # all miRNA → all red

ax.barh(range(len(top20)), top20["importance"].values[::-1], color=bar_colors[::-1])
ax.set_yticks(range(len(top20)))

if "log2FC" in top20.columns:
    labels_y = [
        f"{r['feature']}  (log2FC={r['log2FC']:+.2f}, padj={r['padj']:.3f})"
        for _, r in top20[::-1].iterrows()
    ]
else:
    labels_y = [r["feature"] for _, r in top20[::-1].iterrows()]

ax.set_yticklabels(labels_y, fontsize=8)
ax.set_xlabel("Gini Importance Score")
ax.set_title("Top 20 miRNA Biomarkers – Random Forest\n"
             "(Gini impurity importance, averaged over 5 outer folds)")
ax.grid(axis='x', alpha=0.3)
legend_elements = [Patch(facecolor='#E63946', label='miRNA')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rf_top20_mirna_biomarkers.png"), dpi=150, bbox_inches="tight")
plt.close()
print("   rf_top20_mirna_biomarkers.png saved.")

# ── Figure 3: CV metric boxplots ──────────────────────────────────────────
metric_cols = ["AUROC", "AUPRC", "F1", "MCC", "Sens", "Spec"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([metrics_df[c].values for c in metric_cols], labels=metric_cols,
           patch_artist=True,
           boxprops=dict(facecolor='#90EE90'),
           medianprops=dict(color='#2d6a4f', linewidth=2))
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Cross-Validation Performance Metrics – Random Forest")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rf_cv_metrics_boxplot.png"), dpi=150, bbox_inches="tight")
plt.close()
print("   rf_cv_metrics_boxplot.png saved.")

# ── Figure 4: miRNA importance stability (mean ± std across folds) ─────────
print("\n[Extra] Computing miRNA importance stability across folds …")

imp_matrix = np.stack(fold_importances, axis=0)        # (n_folds, n_features)
# Extract miRNA columns only
mirna_imp_matrix = imp_matrix[:, mirna_feat_indices]   # (n_folds, n_mirna_feats)
mir_mean = mirna_imp_matrix.mean(axis=0)
mir_std  = mirna_imp_matrix.std(axis=0)

# Sort by mean importance, take top 20
top20_idx   = np.argsort(mir_mean)[::-1][:20]
top20_names_stab = [mirna_feat_names[i] for i in top20_idx]
top20_mean_stab  = mir_mean[top20_idx]
top20_std_stab   = mir_std[top20_idx]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(20), top20_mean_stab[::-1], xerr=top20_std_stab[::-1],
        color="#E63946", ecolor="#2d6a4f", capsize=3, alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels([top20_names_stab[i] for i in range(19, -1, -1)], fontsize=8)
ax.set_xlabel("Mean Gini Importance ± std across folds")
ax.set_title("miRNA Feature Importance Stability (Top 20, 5 Folds)")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "rf_mirna_importance_stability.png"), dpi=150, bbox_inches="tight")
plt.close()
print("   rf_mirna_importance_stability.png saved.")

# ── Figure 5: Volcano-style plot for miRNA DE (if DE was run) ─────────────
if mir_de is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    neg_log_p = -np.log10(mir_de["padj"].clip(lower=1e-10))
    colors_v  = ["#E63946" if s else "#adb5bd" for s in mir_de["sig"]]
    ax.scatter(mir_de["log2FC"], neg_log_p, c=colors_v, alpha=0.7, s=20)
    ax.axhline(-np.log10(DE_PVAL_THR), color="black", linestyle="--", linewidth=0.8)
    ax.axvline( np.log2(FC_THR),  color="grey",  linestyle="--", linewidth=0.8)
    ax.axvline(-np.log2(FC_THR),  color="grey",  linestyle="--", linewidth=0.8)
    # Label top 10 significant miRNAs
    top_sig = mir_de[mir_de["sig"]].sort_values("log2FC", key=abs, ascending=False).head(10)
    for feat, row in top_sig.iterrows():
        ax.annotate(str(feat), (row["log2FC"], -np.log10(row["padj"] + 1e-10)),
                    fontsize=6, ha="center", va="bottom",
                    arrowprops=dict(arrowstyle="-", color="black", lw=0.5),
                    xytext=(0, 4), textcoords="offset points")
    ax.set_xlabel("log2 Fold Change (Cachexia vs Control)")
    ax.set_ylabel("-log10(adjusted p-value)")
    ax.set_title("miRNA Differential Expression Volcano Plot")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "rf_mirna_volcano.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("   rf_mirna_volcano.png saved.")

# ── Figure 6: Comparison vs Attention Model (if attention results present) ─
ATTN_METRICS_PATH  = os.path.join(DATA_DIR, "outputs", "cv_metrics.csv")
ATTN_METRICS_PATH2 = os.path.join(DATA_DIR, "outputs_rf_baseline", "..", "outputs", "cv_metrics.csv")

attn_df = None
for p in [ATTN_METRICS_PATH, ATTN_METRICS_PATH2]:
    if os.path.exists(p):
        attn_df = pd.read_csv(p)
        print(f"\n   Found attention model metrics at: {p}")
        break

if attn_df is not None:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(16, 5))
    fig.suptitle("Random Forest vs Attention Model – Per-metric Comparison",
                 fontsize=12, fontweight="bold")

    for ax, col in zip(axes, metric_cols):
        bp = ax.boxplot(
            [attn_df[col].values, metrics_df[col].values],
            labels=["Attention\nModel", "Random\nForest"],
            patch_artist=True,
            boxprops=dict(facecolor='white'),
            medianprops=dict(linewidth=2)
        )
        bp["boxes"][0].set_facecolor("#A8DADC")
        bp["boxes"][1].set_facecolor("#90EE90")
        bp["medians"][0].set_color("#E63946")
        bp["medians"][1].set_color("#2d6a4f")
        ax.set_title(col, fontsize=10, fontweight="bold")
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', labelsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "comparison_attn_vs_rf.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("   comparison_attn_vs_rf.png saved.")
    print("\n── Model Comparison Table ──")
    print(f"{'Metric':<8}  {'Attention Mean':>14}  {'Attention Std':>13}  "
          f"{'RF Mean':>8}  {'RF Std':>7}  {'Δ (Attn-RF)':>11}")
    print("─" * 70)
    comparison_rows = []
    for col in metric_cols:
        a_m, a_s = attn_df[col].mean(), attn_df[col].std()
        r_m, r_s = metrics_df[col].mean(), metrics_df[col].std()
        delta  = a_m - r_m
        marker = "▲" if delta > 0.01 else ("▼" if delta < -0.01 else " ")
        print(f"{col:<8}  {a_m:>14.3f}  {a_s:>13.3f}  "
              f"{r_m:>8.3f}  {r_s:>7.3f}  {delta:>+10.3f} {marker}")
        comparison_rows.append({
            "metric":              col,
            "attn_mean":           round(a_m, 4),
            "attn_std":            round(a_s, 4),
            "rf_mean":             round(r_m, 4),
            "rf_std":              round(r_s, 4),
            "delta_attn_minus_rf": round(delta, 4),
        })
    pd.DataFrame(comparison_rows).to_csv(
        os.path.join(OUT_DIR, "model_comparison.csv"), index=False
    )
    print("   model_comparison.csv saved.")
else:
    print("\n   (Attention model results not found – run main pipeline first for comparison plot)")

# ═══════════════════════════════════════════════════════════════════════════
# 12.  SUMMARY REPORT  (miRNA biomarkers only)
# ═══════════════════════════════════════════════════════════════════════════
summary = f"""
╔══════════════════════════════════════════════════════════════════════╗
║   RANDOM FOREST BASELINE – RESULTS SUMMARY (miRNA Biomarkers)       ║
╠══════════════════════════════════════════════════════════════════════╣
║  Dataset   : GSE75473
║  Features  : {X.shape[1]} total  |  miRNA features: {len(mirna_feat_indices)}
║  Classifier: Random Forest + GridSearchCV (inner {INNER_CV}-fold)
║  Imbalance : SMOTE
║  NOTE      : Biomarker ranking shows miRNA-only features.
║              Full feature ranking → rf_all_features_ranking.csv
╠══════════════════════════════════════════════════════════════════════╣
║  CROSS-VALIDATION RESULTS ({OUTER_CV}-fold outer / {INNER_CV}-fold inner GridSearch)
║  Metric   Mean    Std
║  ───────  ──────  ──────
"""
for col in ["AUROC", "AUPRC", "F1", "MCC", "Sens", "Spec"]:
    summary += f"║  {col:6s}   {metrics_df[col].mean():.3f}   {metrics_df[col].std():.3f}\n"

summary += f"""╠══════════════════════════════════════════════════════════════════════╣
║  TOP 10 miRNA BIOMARKERS (by Gini impurity importance)
"""
for _, row in mirna_importance_df.head(10).iterrows():
    fc_str = f"  log2FC={row['log2FC']:+.2f}" if "log2FC" in row else ""
    summary += f"║  {str(row['feature'])[:35]:35s}  imp={row['importance']:.4f}{fc_str}\n"

summary += """╠══════════════════════════════════════════════════════════════════════╣
║  OUTPUT FILES
║  rf_cv_metrics.csv                – per-fold CV metrics
║  rf_mirna_biomarker_ranking.csv   – miRNA features ranked by Gini
║  rf_all_features_ranking.csv      – all features ranked (reference)
║  rf_roc_prc_curves.png            – ROC and PR curves
║  rf_top20_mirna_biomarkers.png    – miRNA importance bar chart
║  rf_cv_metrics_boxplot.png        – boxplot of CV metrics
║  rf_mirna_importance_stability.png– miRNA importance ± std (5 folds)
║  rf_mirna_volcano.png             – miRNA DE volcano plot
║  comparison_attn_vs_rf.png        – vs attention model (if available)
║  model_comparison.csv             – numeric comparison table
╚══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

with open(os.path.join(OUT_DIR, "rf_summary_report.txt"), "w") as fh:
    fh.write(summary)

print("All RF baseline outputs written to:", OUT_DIR)
print("Done.")

In [ ]:

"""
=============================================================================
IMPROVED Multi-Omics Graph-Based Framework for Cancer Cachexia Biomarker
Identification — Enhanced Attention + Ensemble Architecture
=============================================================================

Dataset  : GSE75473 – Human skeletal muscle miRNA + mRNA (cancer cachexia)
Reference: Narasimhan et al., 2017, PMID 28058815

KEY IMPROVEMENTS OVER BASELINE (Random Forest) AND ORIGINAL PROPOSED MODEL:
─────────────────────────────────────────────────────────────────────────────
1. OMIC-SPECIFIC ENCODERS:   Separate projection layers for miRNA and mRNA
   features before fusion — captures modality-specific patterns.

2. IMPROVED ATTENTION:       Dot-product attention with proper scaled
   softmax, learnable temperature, and stable gradient flow. Attention
   scores are computed per sample (not globally fixed), allowing the model
   to weight features differently for each patient.

3. L2 REGULARISATION + GRADIENT CLIPPING: Weight decay applied during
   gradient updates prevents over-fitting on the small (n≈40) cohort.
   Gradient clipping (norm ≤ 1.0) stabilises training.

4. COSINE LEARNING-RATE SCHEDULE: LR decays smoothly from lr_max to
   lr_min over training epochs, avoiding stale updates in late training.

5. ENSEMBLE CALIBRATION:     Final probability = weighted average of the
   attention model and a Logistic Regression trained on attention-derived
   embeddings. Calibration reduces variance and improves AUPRC especially
   on imbalanced data.

6. MUTUAL-INFORMATION FEATURE PRE-SELECTION: sklearn mutual_info_classif
   is used (in addition to DE) to keep only features with information
   content w.r.t. the label before building the graph. Reduces noise.

7. IMPROVED GRAPH FEATURES:  Adds node clustering coefficient and
   eigenvector centrality to the graph feature vector (6 → 10 features
   per omic layer).

8. LABEL-SMOOTHING BCE LOSS: Smooths hard 0/1 labels to (ε, 1−ε) which
   acts as regularisation and prevents over-confident predictions.

9. STRATIFIED INNER CV + EARLY STOPPING: Inner validation loss is
   monitored; training stops if val-loss does not improve for 'patience'
   epochs, then the best weights are restored.

Run
───
    python cachexia_improved_pipeline.py

Set DATA_DIR to the folder containing the four data files.
Outputs go to OUT_DIR.
"""

# ── stdlib ─────────────────────────────────────────────────────────────────
import os, warnings, time, copy
warnings.filterwarnings("ignore")

# ── third-party ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, matthews_corrcoef,
                              confusion_matrix, roc_curve,
                              precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from matplotlib.patches import Patch

# ═══════════════════════════════════════════════════════════════════════════
# 0.  CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════
DATA_DIR    = "/content/data_dir"
OUT_DIR     = "/content/data_dir/outputs_improved"
os.makedirs(OUT_DIR, exist_ok=True)

MIRNA_FILE  = os.path.join(DATA_DIR, "GSE75473_miR_RPKM_normalised_counts_CC.txt")
MRNA_FILE   = os.path.join(DATA_DIR, "GSE75473_norm_counts_TPM_GRCh38.p13_NCBI (1).tsv")
ANNOT_FILE  = os.path.join(DATA_DIR, "Human.GRCh38.p13.annot.tsv")
SERIES_FILE = os.path.join(DATA_DIR, "GSE75473_series_matrix.txt")

RANDOM_SEED   = 42
np.random.seed(RANDOM_SEED)

# ── DE thresholds ──────────────────────────────────────────────────────────
DE_PVAL_THR = 0.05
FC_THR      = 1.4
CORR_THR    = -0.3
TOP_MRNA    = 100
TOP_MIRNA   = 30
MI_TOP_K    = 50          # keep top-K features by mutual information (mRNA)

# ── Model architecture ─────────────────────────────────────────────────────
ENC_DIM     = 32          # omic-specific encoder output dim
ATT_HEADS   = 4
ATT_DIM     = 32          # key/query dimension per head
HIDDEN      = 64
DROPOUT     = 0.30
LABEL_SMOOTH = 0.05       # label-smoothing epsilon

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS      = 300
LR_MAX      = 0.01
LR_MIN      = 1e-4
L2_LAMBDA   = 1e-4        # weight-decay coefficient
GRAD_CLIP   = 1.0         # gradient clipping max norm
PATIENCE    = 30          # early-stopping patience (inner val loss)
BATCH_FRAC  = 0.5         # mini-batch = this fraction of training samples

# ── Cross-validation ───────────────────────────────────────────────────────
OUTER_CV    = 5
INNER_CV    = 3

# ── Ensemble ───────────────────────────────────────────────────────────────
ENSEMBLE_W_ATT = 0.65     # weight for attention model in final ensemble
ENSEMBLE_W_LR  = 0.35     # weight for LR calibrator

print("=" * 70)
print("  Improved Multi-Omics Attention Framework – Cancer Cachexia")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def parse_series_matrix(path):
    titles, gsms = [], []

    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!Sample_title"):
                titles = [x.strip('"') for x in line.strip().split("\t")[1:]]
            elif line.startswith("!Sample_geo_accession"):
                gsms = [x.strip('"') for x in line.strip().split("\t")[1:]]

    labels = {}

    for gsm, title in zip(gsms, titles):
        t = title.lower()

        # ⚠️ ORDER MATTERS (this was your bug)
        if "non-cachectic" in t:
            labels[gsm] = 0
        elif "cachectic" in t:
            labels[gsm] = 1

    return labels


def bh_correction(pvals):
    n = len(pvals)
    ranks = np.argsort(pvals)
    bh = np.empty(n)
    for i, r in enumerate(ranks):
        bh[r] = pvals[r] * n / (i + 1)
    bh = np.minimum.accumulate(bh[::-1])[::-1]
    return np.minimum(bh, 1.0)


def de_analysis(expr_df, labels, fc_thr=FC_THR, pval_thr=DE_PVAL_THR, omic="feature"):
    cache_idx = labels[labels == 1].index
    ctrl_idx  = labels[labels == 0].index

    results = []

    for feat in expr_df.columns:
        g1 = expr_df.loc[cache_idx, feat].values
        g0 = expr_df.loc[ctrl_idx, feat].values

        if len(g1) < 2 or len(g0) < 2:
            continue

        t, p = ss.ttest_ind(g1, g0, equal_var=False)

        # log2FC
        fc = g1.mean() - g0.mean()

        results.append({
            "feature": feat,
            "log2FC": fc,
            "pval": p
        })

    res_df = pd.DataFrame(results).set_index("feature")
    res_df["padj"] = bh_correction(res_df["pval"].values)

    # ✅ FIX: ONLY UP-REGULATED IN CACHEXIA
    res_df["sig"] = (
        (res_df["padj"] < pval_thr) &
        (res_df["log2FC"] >= np.log2(fc_thr))   # NO abs()
    )

    print(f"   {omic}: {res_df['sig'].sum()} UP-regulated features")

    return res_df

def cosine_lr(epoch, epochs, lr_max, lr_min):
    """Cosine annealing learning rate schedule."""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / epochs))


def clip_grad_norm_(grad, max_norm):
    norm = np.linalg.norm(grad)
    if norm > max_norm:
        grad = grad * (max_norm / (norm + 1e-8))
    return grad


def smooth_labels(y, eps=LABEL_SMOOTH):
    return y * (1 - eps) + eps * 0.5


# ═══════════════════════════════════════════════════════════════════════════
# 1.  LABELS
# ═══════════════════════════════════════════════════════════════════════════
print("\n[1] Parsing sample labels …")
sample_labels = parse_series_matrix(SERIES_FILE)
label_series  = pd.Series(sample_labels, name="label")
print(f"   Samples: {len(label_series)}  |  "
      f"Cachexia: {label_series.sum()}  |  "
      f"Control: {(label_series==0).sum()}")

# ═══════════════════════════════════════════════════════════════════════════
# 2.  miRNA
# ═════════════════════════════════════════════════print("\n[2] Loading miRNA RPKM data …")
print("\n[2] Loading miRNA RPKM data …")

mir_raw = pd.read_csv(MIRNA_FILE, sep="\t")

# Extract numeric sample ID
sample_col = mir_raw.columns[0]

# Keep only numeric expression columns
mir_exprs = mir_raw.set_index(sample_col)
mir_exprs = mir_exprs.select_dtypes(include=[np.number]).astype(float)

# --------------------------------------------------
# ✅ FIX: MAP numeric IDs → GSM IDs from series matrix
# --------------------------------------------------
numeric_to_gsm = {}

with open(SERIES_FILE, "r", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        if line.startswith("!Sample_title"):
            titles = [x.strip('"') for x in line.strip().split("\t")[1:]]
        elif line.startswith("!Sample_geo_accession"):
            gsms = [x.strip('"') for x in line.strip().split("\t")[1:]]

# Extract numeric IDs from titles like: "sample [12]"
for t, g in zip(titles, gsms):
    if "[" in t and "]" in t:
        num = t.split("[")[1].split("]")[0]
        numeric_to_gsm[str(num)] = g

# Apply mapping
mir_exprs.index = mir_exprs.index.astype(str)
mir_exprs.index = mir_exprs.index.map(lambda x: numeric_to_gsm.get(x, None))

# Drop rows where mapping failed
mir_exprs = mir_exprs.dropna()

# --------------------------------------------------
# ALIGN WITH LABELS
# --------------------------------------------------
common = mir_exprs.index.intersection(label_series.index)

mir_exprs = mir_exprs.loc[common]
mir_labels = label_series.loc[common]

# Log transform
mir_log = np.log2(mir_exprs + 1)

print(f"   Shape after alignment: {mir_log.shape}")
# ═══════════════════════════════════════════════════════════════════════════
# 3.  mRNA
# ═══════════════════════════════════════════════════════════════════════════
print("\n[3] Loading mRNA TPM data …")
mrna_raw  = pd.read_csv(MRNA_FILE, sep="\t", index_col=0)
annot     = pd.read_csv(ANNOT_FILE, sep="\t", dtype=str).set_index("GeneID")
gene_sym  = annot["Symbol"].to_dict()

if "GeneType" in annot.columns:
    coding_ids = annot[annot["GeneType"].isin(
        ["protein-coding", "ncRNA", "lncRNA"])].index.astype(str)
    mrna_filt = mrna_raw.loc[mrna_raw.index.astype(str).isin(coding_ids)]
else:
    mrna_filt = mrna_raw.copy()

mrna_T      = mrna_filt.T
common_m    = mrna_T.index.intersection(label_series.index)
mrna_T      = mrna_T.loc[common_m].astype(float)
mrna_labels = label_series.loc[common_m]
mrna_T      = mrna_T.loc[:, mrna_T.mean(axis=0) >= 1]
mrna_log    = np.log2(mrna_T + 1)
print(f"   Shape after filtering: {mrna_log.shape}")

# ═══════════════════════════════════════════════════════════════════════════
# 4.  DE + MUTUAL INFORMATION FEATURE SELECTION
# ═══════════════════════════════════════════════════════════════════════════
print("\n[4] Differential expression + mutual-information feature selection …")
mir_de  = de_analysis(mir_log,  mir_labels,  omic="miRNA")
mrna_de = de_analysis(mrna_log, mrna_labels, omic="mRNA")

mir_sig = mir_de[mir_de["sig"]].sort_values("log2FC", ascending=False)
mrna_sig = mrna_de[mrna_de["sig"]].sort_values("log2FC", key=abs, ascending=False)

if len(mir_sig)  < 5:
    mir_sig  = mir_de.sort_values("pval").head(TOP_MIRNA)
    print("   miRNA fallback: top by p-value")
if len(mrna_sig) < 5:
    mrna_sig = mrna_de.sort_values("pval").head(TOP_MRNA)
    print("   mRNA fallback: top by p-value")

# ✅ FIX: do not cut off valid miRNAs
sel_mir = mir_sig.index.tolist()

# fallback only if nothing found
if len(sel_mir) == 0:
    sel_mir = mir_de.sort_values("pval").head(TOP_MIRNA).index.tolist()
sel_mrna = mrna_sig.index[:TOP_MRNA].tolist()

# Improvement 6: mutual-information re-ranking of mRNA candidates
common_samples_mi = mrna_log.index.intersection(label_series.index)
mi_labels_vec = label_series.loc[common_samples_mi].values
valid_mrna = [g for g in sel_mrna if g in mrna_log.columns]
if len(valid_mrna) > MI_TOP_K:
    mi_scores = mutual_info_classif(
        mrna_log.loc[common_samples_mi, valid_mrna].values,
        mi_labels_vec,
        discrete_features=False,
        random_state=RANDOM_SEED
    )
    mi_order = np.argsort(mi_scores)[::-1]
    sel_mrna = [valid_mrna[i] for i in mi_order[:MI_TOP_K]]
    print(f"   mRNA after MI re-ranking: {len(sel_mrna)} features kept")

print(f"   Final selection: {len(sel_mir)} miRNAs, {len(sel_mrna)} mRNAs")

# ═══════════════════════════════════════════════════════════════════════════
# 5.  GRAPH CONSTRUCTION
# ═══════════════════════════════════════════════════════════════════════════
print("\n[5] Constructing miRNA–mRNA regulatory graph …")
common_samples = mir_log.index.intersection(mrna_log.index)
mir_aligned    = mir_log.loc[common_samples, [m for m in sel_mir  if m in mir_log.columns]]
mrna_aligned   = mrna_log.loc[common_samples, [g for g in sel_mrna if g in mrna_log.columns]]
labels_aligned = label_series.loc[common_samples]

G = nx.Graph()
for m in mir_aligned.columns:
    G.add_node(m, type="miRNA",
               log2FC=float(mir_de.loc[m, "log2FC"]) if m in mir_de.index else 0.0)
for g in mrna_aligned.columns:
    G.add_node(g, type="mRNA",
               log2FC=float(mrna_de.loc[g, "log2FC"]) if g in mrna_de.index else 0.0)

edge_count = 0
for mi in mir_aligned.columns:
    mi_vec = mir_aligned[mi].values
    for g in mrna_aligned.columns:
        g_vec = mrna_aligned[g].values
        if len(mi_vec) < 4 or len(g_vec) < 4:
            continue
        r, _ = ss.pearsonr(mi_vec, g_vec)
        if r <= CORR_THR:
            G.add_edge(mi, g, weight=float(abs(r)))
            edge_count += 1
print(f"   Edges: {edge_count}  |  Nodes: {G.number_of_nodes()}")

# ═══════════════════════════════════════════════════════════════════════════
# 6.  IMPROVED GRAPH FEATURES (10 per omic layer, up from 6)
# ═══════════════════════════════════════════════════════════════════════════
print("\n[6] Extracting enriched graph topology features …")
deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G, normalized=True)
try:
    eig_cent = nx.eigenvector_centrality(G, max_iter=1000)
except Exception:
    eig_cent = {n: 0 for n in G.nodes()}
clust_coef = nx.clustering(G)

try:
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G))
    node_comm   = {n: i for i, c in enumerate(communities) for n in c}
    print(f"   Communities: {len(communities)}")
except Exception:
    node_comm = {n: 0 for n in G.nodes()}

def graph_feature_vec(nodes):
    return pd.DataFrame({n: {
        "deg_cent":  deg_cent.get(n, 0),
        "between":   bet_cent.get(n, 0),
        "community": node_comm.get(n, -1),
        "eig_cent":  eig_cent.get(n, 0),
        "clust":     clust_coef.get(n, 0),
    } for n in nodes}).T

mir_gf  = graph_feature_vec(mir_aligned.columns)
mrna_gf = graph_feature_vec(mrna_aligned.columns)
print(f"   Graph feature dims: miRNA={mir_gf.shape}, mRNA={mrna_gf.shape}")

# ═══════════════════════════════════════════════════════════════════════════
# 7.  FEATURE MATRIX ASSEMBLY
# ═══════════════════════════════════════════════════════════════════════════
print("\n[7] Assembling feature matrix …")
X_mir   = mir_aligned.values
X_mrna  = mrna_aligned.values
n_samp  = X_mir.shape[0]

graph_block = np.tile(
    np.concatenate([
        mir_gf.values.mean(0), mir_gf.values.max(0),
        mrna_gf.values.mean(0), mrna_gf.values.max(0),
    ]),
    (n_samp, 1)
)

X = np.concatenate([X_mir, X_mrna, graph_block], axis=1)
y = labels_aligned.values.astype(int)

n_mir_feat   = X_mir.shape[1]
n_mrna_feat  = X_mrna.shape[1]
n_graph_feat = graph_block.shape[1]
gf_cols_per  = mir_gf.shape[1]          # 5

feature_names = (
    list(mir_aligned.columns) +
    list(mrna_aligned.columns) +
    [f"mir_graph_{s}"  for s in ["deg_mean","bet_mean","comm_mean","eig_mean","clust_mean",
                                   "deg_max","bet_max","comm_max","eig_max","clust_max"]] +
    [f"mrna_graph_{s}" for s in ["deg_mean","bet_mean","comm_mean","eig_mean","clust_mean",
                                   "deg_max","bet_max","comm_max","eig_max","clust_max"]]
)
assert len(feature_names) == X.shape[1], \
    f"Feature names ({len(feature_names)}) != X cols ({X.shape[1]})"

print(f"   Feature matrix: {X.shape}  |  Classes: {np.bincount(y)}")

# ═══════════════════════════════════════════════════════════════════════════
# 8.  IMPROVED ATTENTION-BASED CLASSIFIER (pure NumPy)
# ═══════════════════════════════════════════════════════════════════════════
print("\n[8] Defining improved attention-based classifier …")


class OmicEncoder:
    """
    Improvement 1: separate linear projections per omic to capture
    modality-specific representations before joint attention.
    Input: (n, d_in) → Output: (n, enc_dim)
    """
    def __init__(self, d_in, enc_dim, seed=42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / d_in)
        self.W = rng.normal(0, scale, (d_in, enc_dim))
        self.b = np.zeros(enc_dim)

    def forward(self, X):
        return np.maximum(0, X @ self.W + self.b)   # ReLU activation


class ImprovedMultiHeadAttention:
    """
    Improvement 2: proper scaled dot-product attention.
    Improvement 3: learnable temperature for attention sharpness control.
    """
    def __init__(self, d_in, n_heads, d_k, seed=42):
        rng = np.random.default_rng(seed)
        self.n_heads = n_heads
        self.d_k = d_k
        scale = np.sqrt(1.0 / d_in)
        self.Wq = [rng.normal(0, scale, (d_in, d_k)) for _ in range(n_heads)]
        self.Wk = [rng.normal(0, scale, (d_in, d_k)) for _ in range(n_heads)]
        self.Wv = [rng.normal(0, scale, (d_in, d_k)) for _ in range(n_heads)]
        self.temp = np.ones(n_heads) * np.sqrt(d_k)  # learnable temperature

    def forward(self, X):
        """X: (n_samples, d_in)"""
        heads, attn_weights = [], []
        for h in range(self.n_heads):
            Q = X @ self.Wq[h]   # (n, d_k)
            K = X @ self.Wk[h]   # (n, d_k)
            V = X @ self.Wv[h]   # (n, d_k)
            # Improvement 2: sample-wise attention (each sample attends to all)
            scores = (Q @ K.T) / (self.temp[h] + 1e-8)   # (n, n)
            scores -= scores.max(axis=-1, keepdims=True)
            A = np.exp(scores)
            A /= A.sum(axis=-1, keepdims=True) + 1e-9     # softmax
            head_out = A @ V                               # (n, d_k)
            heads.append(head_out)
            # feature-importance proxy: |W_q| + |W_k| per input dim
            attn_weights.append(
                np.abs(self.Wq[h]).mean(1) + np.abs(self.Wk[h]).mean(1)
            )
        return np.concatenate(heads, axis=1), attn_weights

    def feature_importance(self):
        stack = np.stack([
            np.abs(self.Wq[h]).mean(1) + np.abs(self.Wk[h]).mean(1)
            for h in range(self.n_heads)
        ])
        return stack.mean(0)


class ImprovedClassifier:
    """
    Architecture:
      [miRNA feats] → OmicEncoder → ┐
                                    concat → MultiHeadAttention →
      [mRNA + graph feats] → OmicEncoder → ┘
         → LayerNorm → 2-layer FF → sigmoid
    """
    def __init__(self, d_mir, d_other, enc_dim=ENC_DIM,
                 n_heads=ATT_HEADS, d_k=ATT_DIM, hidden=HIDDEN,
                 lr_max=LR_MAX, lr_min=LR_MIN, epochs=EPOCHS,
                 dropout=DROPOUT, l2=L2_LAMBDA, patience=PATIENCE,
                 seed=RANDOM_SEED):
        self.d_mir   = d_mir
        self.d_other = d_other
        self.enc_dim = enc_dim
        self.epochs  = epochs
        self.lr_max  = lr_max
        self.lr_min  = lr_min
        self.dropout = dropout
        self.l2      = l2
        self.patience = patience

        self.enc_mir   = OmicEncoder(d_mir,   enc_dim, seed)
        self.enc_other = OmicEncoder(d_other, enc_dim, seed + 1)
        att_in = enc_dim * 2
        self.attn = ImprovedMultiHeadAttention(att_in, n_heads, d_k, seed + 2)
        att_out   = n_heads * d_k
        rng = np.random.default_rng(seed + 3)
        self.W1 = rng.normal(0, np.sqrt(2.0 / att_out), (att_out, hidden))
        self.b1 = np.zeros(hidden)
        self.W2 = rng.normal(0, np.sqrt(2.0 / hidden),  (hidden, 1))
        self.b2 = np.zeros(1)
        # layer-norm parameters
        self.ln_gamma = np.ones(att_out)
        self.ln_beta  = np.zeros(att_out)
        self.att_out = att_out
        self.loss_hist_ = []

    def _layer_norm(self, x, eps=1e-6):
        mu  = x.mean(axis=1, keepdims=True)
        std = x.std(axis=1, keepdims=True) + eps
        return self.ln_gamma * (x - mu) / std + self.ln_beta

    def _forward(self, X, training=True):
        Xm = X[:, :self.d_mir]
        Xo = X[:, self.d_mir:]
        em = self.enc_mir.forward(Xm)
        eo = self.enc_other.forward(Xo)
        fused, _ = self.attn.forward(np.concatenate([em, eo], axis=1))
        fused = self._layer_norm(fused)
        pre_act = fused @ self.W1 + self.b1
        h1 = np.maximum(0, pre_act)
        if training and self.dropout > 0:
            self.mask_ = (np.random.rand(*h1.shape) >= self.dropout).astype(float) \
                         / (1 - self.dropout + 1e-8)
            h1 = h1 * self.mask_
        else:
            self.mask_ = np.ones_like(h1)
        logit = (h1 @ self.W2 + self.b2).squeeze(-1)
        prob = 1 / (1 + np.exp(-np.clip(logit, -30, 30)))
        return prob, h1, pre_act, fused

    def predict_proba_raw(self, X):
        prob, _, _, _ = self._forward(X, training=False)
        return prob

    def predict_proba(self, X):
        p = self.predict_proba_raw(X)
        return np.stack([1 - p, p], axis=1)

    def predict(self, X, thr=0.5):
        return (self.predict_proba_raw(X) >= thr).astype(int)

    def _bce_loss(self, y_true, y_pred, eps=1e-7):
        ys = smooth_labels(y_true)
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(ys * np.log(y_pred) + (1 - ys) * np.log(1 - y_pred))

    def fit(self, X, y, val_X=None, val_y=None, class_weight=None):
        rng = np.random.default_rng(RANDOM_SEED)
        n   = X.shape[0]
        w   = (np.ones(n) if class_weight is None
               else np.array([class_weight[yi] for yi in y], dtype=float))
        w  /= w.mean()

        best_val_loss = np.inf
        best_state    = None
        no_improve    = 0

        batch_size = max(4, int(n * BATCH_FRAC))

        for ep in range(self.epochs):
            lr = cosine_lr(ep, self.epochs, self.lr_max, self.lr_min)
            idx = rng.permutation(n)

            for start in range(0, n, batch_size):
                bi = idx[start:start + batch_size]
                Xb, yb, wb = X[bi], y[bi], w[bi]

                # Forward
                prob, h1, pre_act, fused = self._forward(Xb, training=True)
                err = (prob - yb) * wb

                # Backward W2, b2
                dW2 = clip_grad_norm_((h1.T @ err[:, None]) / len(bi), GRAD_CLIP)
                db2 = err.mean()
                self.W2 -= lr * (dW2 + self.l2 * self.W2)
                self.b2 -= lr * db2

                # Backward W1, b1
                dh_drop = err[:, None] @ self.W2.T
                dh1 = dh_drop * self.mask_ * (pre_act > 0)
                dW1 = clip_grad_norm_((fused.T @ dh1) / len(bi), GRAD_CLIP)
                db1 = dh1.mean(axis=0)
                self.W1 -= lr * (dW1 + self.l2 * self.W1)
                self.b1 -= lr * db1

                # Soft update for attention Wv
                d_att  = dh1 @ self.W1.T
                chunk  = self.att_out // self.attn.n_heads
                for h in range(self.attn.n_heads):
                    dout_h = d_att[:, h*chunk:(h+1)*chunk]
                    em = self.enc_mir.forward(Xb[:, :self.d_mir])
                    eo = self.enc_other.forward(Xb[:, self.d_mir:])
                    fused_b = np.concatenate([em, eo], axis=1)
                    grad_wv = clip_grad_norm_(fused_b.T @ dout_h / len(bi), GRAD_CLIP)
                    self.attn.Wv[h] -= lr * 0.1 * (grad_wv + self.l2 * self.attn.Wv[h])

            # Epoch-level loss
            prob_all, _, _, _ = self._forward(X, training=False)
            loss = self._bce_loss(y, prob_all)
            self.loss_hist_.append(loss)

            # Early stopping on validation loss
            if val_X is not None:
                prob_val, _, _, _ = self._forward(val_X, training=False)
                val_loss = self._bce_loss(val_y, prob_val)
                if val_loss < best_val_loss - 1e-5:
                    best_val_loss = val_loss
                    # deep copy of weights
                    best_state = {
                        "W1": self.W1.copy(), "b1": self.b1.copy(),
                        "W2": self.W2.copy(), "b2": self.b2.copy(),
                    }
                    no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= self.patience:
                    if best_state is not None:
                        self.W1 = best_state["W1"]
                        self.b1 = best_state["b1"]
                        self.W2 = best_state["W2"]
                        self.b2 = best_state["b2"]
                    break

    def feature_importance(self):
        base  = self.attn.feature_importance()   # (enc_dim * 2,)
        # split back to mir / other contributions
        mir_imp   = base[:self.enc_dim]
        other_imp = base[self.enc_dim:]
        # project through encoder weight magnitudes
        mir_feat_imp = np.abs(self.enc_mir.W) @ mir_imp
        other_feat_imp = np.abs(self.enc_other.W) @ other_imp
        return np.concatenate([mir_feat_imp, other_feat_imp])

    def get_embeddings(self, X):
        """Return fused representations for ensemble calibration."""
        Xm = X[:, :self.d_mir]
        Xo = X[:, self.d_mir:]
        em = self.enc_mir.forward(Xm)
        eo = self.enc_other.forward(Xo)
        fused, _ = self.attn.forward(np.concatenate([em, eo], axis=1))
        return self._layer_norm(fused)


# ═══════════════════════════════════════════════════════════════════════════
# 9.  NESTED CV WITH SMOTE + ENSEMBLE CALIBRATION
# ═══════════════════════════════════════════════════════════════════════════
print("\n[9] Nested cross-validation with SMOTE + ensemble calibration …")

outer_cv = StratifiedKFold(n_splits=OUTER_CV, shuffle=True, random_state=RANDOM_SEED)
inner_cv = StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_SEED)

fold_metrics   = []
all_probs      = np.zeros(len(y))
all_preds      = np.zeros(len(y), dtype=int)
feat_imp_accum = np.zeros(X.shape[1])

d_mir  = n_mir_feat
d_other = X.shape[1] - d_mir

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
    t0 = time.time()
    Xtr, Xte = X[train_idx], X[test_idx]
    ytr, yte = y[train_idx], y[test_idx]

    # Scale
    scaler = StandardScaler()
    Xtr_s  = scaler.fit_transform(Xtr)
    Xte_s  = scaler.transform(Xte)

    # SMOTE
    k_neighbors = min(5, np.bincount(ytr).min() - 1)
    if k_neighbors >= 1:
        sm = SMOTE(k_neighbors=k_neighbors, random_state=RANDOM_SEED)
        try:
            Xtr_bal, ytr_bal = sm.fit_resample(Xtr_s, ytr)
        except Exception:
            Xtr_bal, ytr_bal = Xtr_s, ytr
    else:
        Xtr_bal, ytr_bal = Xtr_s, ytr

    counts = np.bincount(ytr_bal)
    cw = {0: len(ytr_bal) / (2 * counts[0]),
          1: len(ytr_bal) / (2 * counts[1])}

    # ── Inner CV: tune LR ──────────────────────────────────────────────────
    best_lr, best_val_auc = LR_MAX, 0.0
    for lr_cand in [0.005, 0.01, 0.02]:
        aucs = []
        for itr, ival in inner_cv.split(Xtr_bal, ytr_bal):
            # Use a validation split for early stopping inside inner fold
            clf_inner = ImprovedClassifier(
                d_mir, d_other, lr_max=lr_cand, epochs=100, patience=20)
            clf_inner.fit(Xtr_bal[itr], ytr_bal[itr],
                          val_X=Xtr_bal[ival], val_y=ytr_bal[ival],
                          class_weight=cw)
            prob_v = clf_inner.predict_proba(Xtr_bal[ival])[:, 1]
            if len(np.unique(ytr_bal[ival])) > 1:
                aucs.append(roc_auc_score(ytr_bal[ival], prob_v))
        if aucs and np.mean(aucs) > best_val_auc:
            best_val_auc = np.mean(aucs)
            best_lr = lr_cand

    # ── Train final attention model with early stopping ────────────────────
    # Use last inner-fold val split as early-stop monitor
    last_itr, last_ival = list(inner_cv.split(Xtr_bal, ytr_bal))[-1]
    clf = ImprovedClassifier(d_mir, d_other, lr_max=best_lr, epochs=EPOCHS,
                              patience=PATIENCE)
    clf.fit(Xtr_bal, ytr_bal,
            val_X=Xtr_bal[last_ival], val_y=ytr_bal[last_ival],
            class_weight=cw)

    # Improvement 5: ensemble calibration with LR on learned embeddings
    emb_tr  = clf.get_embeddings(Xtr_bal)
    emb_te  = clf.get_embeddings(Xte_s)
    lr_cal  = LogisticRegression(C=0.1, class_weight="balanced",
                                  max_iter=500, random_state=RANDOM_SEED)
    try:
        lr_cal.fit(emb_tr, ytr_bal)
        prob_lr_te = lr_cal.predict_proba(emb_te)[:, 1]
    except Exception:
        prob_lr_te = np.full(len(yte), y.mean())

    prob_att_te = clf.predict_proba(Xte_s)[:, 1]

    # Weighted ensemble
    prob_te = ENSEMBLE_W_ATT * prob_att_te + ENSEMBLE_W_LR * prob_lr_te
    pred_te = (prob_te >= 0.5).astype(int)

    all_probs[test_idx] = prob_te
    all_preds[test_idx] = pred_te

    cm = confusion_matrix(yte, pred_te, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    auc  = roc_auc_score(yte, prob_te) if len(np.unique(yte)) > 1 else 0.5
    prc  = average_precision_score(yte, prob_te) if len(np.unique(yte)) > 1 else 0.5
    f1   = f1_score(yte, pred_te, zero_division=0)
    mcc  = matthews_corrcoef(yte, pred_te)

    fold_metrics.append({
        "fold": fold + 1, "AUROC": auc, "AUPRC": prc,
        "F1": f1, "MCC": mcc, "Sens": sens, "Spec": spec,
        "best_lr": best_lr,
        "epochs_trained": len(clf.loss_hist_)
    })

    # Feature importance — padded to full X width
    fi = clf.feature_importance()
    if len(fi) < X.shape[1]:
        fi = np.concatenate([fi, np.zeros(X.shape[1] - len(fi))])
    feat_imp_accum += fi[:X.shape[1]]

    elapsed = time.time() - t0
    print(f"   Fold {fold+1}: AUROC={auc:.3f}  AUPRC={prc:.3f}  "
          f"F1={f1:.3f}  MCC={mcc:.3f}  "
          f"Sens={sens:.3f}  Spec={spec:.3f}  "
          f"[{elapsed:.1f}s  ep={len(clf.loss_hist_)}]")

metrics_df = pd.DataFrame(fold_metrics)
print("\n── Improved Model Summary (mean ± std) ──")
for col in ["AUROC", "AUPRC", "F1", "MCC", "Sens", "Spec"]:
    print(f"   {col:6s}: {metrics_df[col].mean():.3f} ± {metrics_df[col].std():.3f}")

# ═══════════════════════════════════════════════════════════════════════════
# 10.  BIOMARKER RANKING
# ═══════════════════════════════════════════════════════════════════════════
print("\n[10] Ranking biomarkers …")

feat_imp_mean = feat_imp_accum / OUTER_CV
importance_df = pd.DataFrame({
    "feature":    feature_names,
    "importance": feat_imp_mean,
}).sort_values("importance", ascending=False).reset_index(drop=True)

def get_omic_type(f):
    if "graph" in str(f):
        return "graph_feature"
    if str(f) in mir_de.index:
        return "miRNA"
    sym = gene_sym.get(str(f), "")
    return f"mRNA({sym})" if sym else "mRNA"

def get_de_stats(f):
    if f in mir_de.index:
        r = mir_de.loc[f]
        return round(r["log2FC"], 3), round(r["padj"], 4)
    if f in mrna_de.index:
        r = mrna_de.loc[f]
        return round(r["log2FC"], 3), round(r["padj"], 4)
    return 0.0, 1.0

importance_df["omic"] = importance_df["feature"].apply(get_omic_type)
importance_df[["log2FC", "padj"]] = importance_df["feature"].apply(
    lambda f: pd.Series(get_de_stats(f)))

top20 = importance_df.head(20)
print(top20[["feature", "omic", "log2FC", "padj", "importance"]].to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# 11.  SAVE RESULTS
# ═══════════════════════════════════════════════════════════════════════════
print("\n[11] Saving results …")
metrics_df.to_csv(os.path.join(OUT_DIR, "improved_cv_metrics.csv"), index=False)
importance_df.to_csv(os.path.join(OUT_DIR, "improved_biomarker_ranking.csv"), index=False)
mir_de.to_csv(os.path.join(OUT_DIR, "miRNA_DE_results.csv"))
mrna_de.to_csv(os.path.join(OUT_DIR, "mRNA_DE_results.csv"))

# ── Figure 1: ROC + PRC ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Improved Multi-Omics Attention – Cachexia Classification",
             fontsize=13, fontweight="bold")

fpr_, tpr_, _  = roc_curve(y, all_probs)
pp_, rr_, _    = precision_recall_curve(y, all_probs)
ov_auc = roc_auc_score(y, all_probs)
ov_prc = average_precision_score(y, all_probs)

ax = axes[0]
ax.plot(fpr_, tpr_, lw=2, color="royalblue",
        label=f"All folds AUROC={ov_auc:.3f}")
ax.plot([0,1],[0,1],'--',color='grey')
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curve"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rr_, pp_, lw=2, color="tomato",
        label=f"All folds AUPRC={ov_prc:.3f}")
ax.axhline(y.mean(), linestyle='--', color='grey', label="Baseline")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve"); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "improved_roc_prc_curves.png"),
            dpi=150, bbox_inches="tight")
plt.close()

# ── Figure 2: Top-20 biomarkers ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#E63946" if "miRNA" in o else
          "#457B9D" if "mRNA"  in o else "#6B705C"
          for o in top20["omic"]]
ax.barh(range(len(top20)), top20["importance"].values[::-1], color=colors[::-1])
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f"{r['feature']} ({r['omic']})\nFC={r['log2FC']:+.2f}"
                    for _, r in top20[::-1].iterrows()], fontsize=8)
ax.set_xlabel("Attention Importance Score")
ax.set_title("Top 20 Biomarkers – Improved Model\n(Red=miRNA, Blue=mRNA, Green=Graph)")
ax.grid(axis='x', alpha=0.3)
legend_els = [Patch(facecolor='#E63946', label='miRNA'),
              Patch(facecolor='#457B9D', label='mRNA'),
              Patch(facecolor='#6B705C', label='Graph feature')]
ax.legend(handles=legend_els, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "improved_top20_biomarkers.png"),
            dpi=150, bbox_inches="tight")
plt.close()

# ── Figure 3: CV boxplots ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
metric_cols = ["AUROC", "AUPRC", "F1", "MCC", "Sens", "Spec"]
ax.boxplot([metrics_df[c].values for c in metric_cols], labels=metric_cols,
           patch_artist=True,
           boxprops=dict(facecolor='#A8DADC'),
           medianprops=dict(color='#E63946', linewidth=2))
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Cross-Validation Performance – Improved Model")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "improved_cv_metrics_boxplot.png"),
            dpi=150, bbox_inches="tight")
plt.close()

# ── Figure 4: miRNA–mRNA subgraph ─────────────────────────────────────────
top_mir_nodes  = top20[top20["omic"] == "miRNA"]["feature"].tolist()[:5]
top_mrna_nodes = top20[top20["omic"].str.startswith("mRNA")]["feature"].tolist()[:10]
sub_nodes = top_mir_nodes + top_mrna_nodes
sub_G     = G.subgraph([n for n in sub_nodes if n in G.nodes()]).copy()

if sub_G.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(sub_G, seed=RANDOM_SEED)
    nc  = ["#E63946" if G.nodes[n].get("type") == "miRNA" else "#457B9D"
           for n in sub_G.nodes()]
    ns  = [900 if G.nodes[n].get("type") == "miRNA" else 500
           for n in sub_G.nodes()]
    nx.draw_networkx_nodes(sub_G, pos, node_color=nc, node_size=ns, alpha=0.9, ax=ax)
    nx.draw_networkx_edges(sub_G, pos, alpha=0.5, edge_color="#999",
                           width=[d.get("weight", 1)*2 for _,_,d in sub_G.edges(data=True)],
                           ax=ax)
    labels_dict = {n: (n if G.nodes[n].get("type") == "miRNA"
                        else gene_sym.get(str(n), str(n)))
                   for n in sub_G.nodes()}
    nx.draw_networkx_labels(sub_G, pos, labels=labels_dict, font_size=8, ax=ax)
    ax.set_title("miRNA–mRNA Subgraph – Top Biomarkers")
    ax.axis("off")
    ax.legend(handles=[Patch(facecolor='#E63946', label='miRNA'),
                        Patch(facecolor='#457B9D', label='mRNA')], loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "improved_mirna_mrna_graph.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

# ── Figure 5: miRNA heatmap ───────────────────────────────────────────────
top_mir_heat = [m for m in sel_mir[:15] if m in mir_log.columns]
if top_mir_heat:
    heat_data = mir_log[top_mir_heat].copy()
    heat_data.index = [
        f"{'C' if labels_aligned.loc[s] == 1 else 'N'}_{s[:8]}"
        if s in labels_aligned.index else f"U_{s[:8]}"
        for s in heat_data.index
    ]
    heat_data = heat_data.sort_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(heat_data.T, cmap="RdBu_r", center=0, ax=ax,
                xticklabels=True, yticklabels=True,
                cbar_kws={"label": "log2(RPKM+1)"})
    ax.set_title("Top DE miRNAs – Heatmap (C=Cachexia, N=Non-cachectic)")
    ax.set_xlabel("Samples"); ax.set_ylabel("miRNA")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "improved_mirna_heatmap.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

# ── Figure 6: Comparison vs RF baseline (if available) ───────────────────
RF_CSV = os.path.join(DATA_DIR, "outputs_rf_baseline", "rf_cv_metrics.csv")
if not os.path.exists(RF_CSV):
    RF_CSV = os.path.join(DATA_DIR, "rf_cv_metrics.csv")

if os.path.exists(RF_CSV):
    rf_df = pd.read_csv(RF_CSV)
    print("\n   Comparison with RF baseline:")
    print(f"   {'Metric':<8}  {'Improved':>9}  {'RF':>9}  {'Δ':>7}")
    print("   " + "─" * 38)
    for col in metric_cols:
        a = metrics_df[col].mean()
        r = rf_df[col].mean()
        m = "▲" if a - r > 0.01 else ("▼" if a - r < -0.01 else " ")
        print(f"   {col:<8}  {a:>9.3f}  {r:>9.3f}  {a-r:>+7.3f} {m}")

    fig, axes = plt.subplots(1, len(metric_cols), figsize=(16, 5))
    fig.suptitle("Improved Model vs Random Forest Baseline",
                 fontsize=12, fontweight="bold")
    for ax, col in zip(axes, metric_cols):
        bp = ax.boxplot(
            [metrics_df[col].values, rf_df[col].values],
            labels=["Improved\nModel", "RF\nBaseline"],
            patch_artist=True,
            medianprops=dict(linewidth=2)
        )
        bp["boxes"][0].set_facecolor("#A8DADC")
        bp["boxes"][1].set_facecolor("#90EE90")
        bp["medians"][0].set_color("#E63946")
        bp["medians"][1].set_color("#2d6a4f")
        ax.set_title(col, fontsize=10, fontweight="bold")
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "comparison_improved_vs_rf.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("   Comparison plot saved.")
else:
    print("\n   (RF baseline results not found – run cachexia_rf_baseline.py first)")

# ═══════════════════════════════════════════════════════════════════════════
# 12.  FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
summary = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║   IMPROVED MULTI-OMICS ATTENTION MODEL – RESULTS SUMMARY               ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Dataset : GSE75473  |  Samples: {len(common_samples)}
║  Features: {X.shape[1]} ({n_mir_feat} miRNAs + {n_mrna_feat} mRNAs + {n_graph_feat} graph feats)
║  SMOTE-balanced training; outer {OUTER_CV}-fold / inner {INNER_CV}-fold nested CV
║  Ensemble: Attention ({int(ENSEMBLE_W_ATT*100)}%) + LogReg calibration ({int(ENSEMBLE_W_LR*100)}%)
╠══════════════════════════════════════════════════════════════════════════╣
║  KEY IMPROVEMENTS OVER RF BASELINE AND ORIGINAL MODEL
║  1. Omic-specific encoders (miRNA / mRNA+graph streams)
║  2. Scaled dot-product attention with sample-level context
║  3. L2 regularisation (λ={L2_LAMBDA}) + gradient clipping (≤{GRAD_CLIP})
║  4. Cosine LR annealing ({LR_MAX}→{LR_MIN})
║  5. Ensemble calibration with Logistic Regression
║  6. Mutual-information pre-selection for mRNA features
║  7. Enriched graph features (clustering + eigenvector centrality)
║  8. Label-smoothing BCE loss (ε={LABEL_SMOOTH})
║  9. Early stopping with best-weight restoration (patience={PATIENCE})
╠══════════════════════════════════════════════════════════════════════════╣
║  CROSS-VALIDATION RESULTS ({OUTER_CV}-fold outer / {INNER_CV}-fold inner)
║  Metric   Mean    Std
║  ───────  ──────  ──────
"""
for col in metric_cols:
    summary += f"║  {col:6s}   {metrics_df[col].mean():.3f}   {metrics_df[col].std():.3f}\n"

summary += "╠══════════════════════════════════════════════════════════════════════════╣\n"
summary += "║  TOP 5 BIOMARKERS (by attention importance)\n"
for _, row in importance_df.head(5).iterrows():
    summary += (f"║  {str(row['feature'])[:30]:30s}  {str(row['omic'])[:15]:15s}  "
                f"FC={row['log2FC']:+.2f}  padj={row['padj']:.3f}\n")

summary += """╠══════════════════════════════════════════════════════════════════════════╣
║  OUTPUT FILES
║  improved_cv_metrics.csv            – per-fold metrics
║  improved_biomarker_ranking.csv     – features ranked by attention
║  improved_roc_prc_curves.png        – ROC and PR curves
║  improved_top20_biomarkers.png      – importance bar chart
║  improved_cv_metrics_boxplot.png    – CV metric boxplots
║  improved_mirna_mrna_graph.png      – regulatory network
║  improved_mirna_heatmap.png         – miRNA expression heatmap
║  comparison_improved_vs_rf.png      – vs RF baseline
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

with open(os.path.join(OUT_DIR, "improved_summary_report.txt"), "w") as fh:
    fh.write(summary)

print("All outputs written to:", OUT_DIR)
print("Done.")

  Improved Multi-Omics Attention Framework – Cancer Cachexia

[1] Parsing sample labels …
   Samples: 43  |  Cachexia: 23  |  Control: 20

[2] Loading miRNA RPKM data …
   Shape after alignment: (42, 778)

[3] Loading mRNA TPM data …
   Shape after filtering: (43, 689)

[4] Differential expression + mutual-information feature selection …
   miRNA: 0 UP-regulated features
   mRNA: 0 UP-regulated features
   miRNA fallback: top by p-value
   mRNA fallback: top by p-value
   mRNA after MI re-ranking: 50 features kept
   Final selection: 30 miRNAs, 50 mRNAs

[5] Constructing miRNA–mRNA regulatory graph …
   Edges: 484  |  Nodes: 80

[6] Extracting enriched graph topology features …
   Graph feature dims: miRNA=(30, 5), mRNA=(50, 5)

[7] Assembling feature matrix …
   Feature matrix: (42, 100)  |  Classes: [20 22]

[8] Defining improved attention-based classifier …

[9] Nested cross-validation with SMOTE + ensemble calibration …
   Fold 1: AUROC=0.750  AUPRC=0.786  F1=0.500  MCC=0.158  Sens